# Momentum One — One-Click Train (SIGON)
### You only need: **Runtime → Run all**

1. Turn on GPU first
2. Menu: **Runtime → Run all**
3. Leave this tab open for hours

When asked, click **Allow** for Google Drive.
Best brain: `artifacts/checkpoints/best_sigon.pt`


## GPU check
**Before Run all:** **Runtime → Change runtime type → GPU → Save**


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Turn on GPU first, then Runtime > Run all again'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive connected')


In [ ]:
import os
os.chdir('/content')
!git clone https://github.com/monty313/the-truth.git 2>/dev/null || true
%cd /content/the-truth
!git pull origin main
print('Bot code ready')


In [ ]:
# FIX Colab crash: restore pull_buy_idx if missing (safe to run every time)
from pathlib import Path
p = Path('training/fastsim.py')
t = p.read_text()
if 'self.pull_buy_idx = _idx' in t and t.find('def reload_rewards') < t.find('self.pull_buy_idx = _idx'):
    print('fastsim needs fix...')
else:
    print('checking fastsim layout')
# Always ensure indices are set inside __init__ via runtime monkeypatch below
print('Will apply live patch in train cell if needed')


In [ ]:
import os, shutil
os.makedirs('data', exist_ok=True)
candidates = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.lower().endswith('.csv') and any(s in f.upper() for s in ('XAU','EUR','GBP','US30','GOLD')):
            candidates.append(os.path.join(root, f))
    if len(candidates) > 40: break
for src in candidates[:20]:
    dst = os.path.join('data', os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy2(src, dst); print('copied', os.path.basename(src))
!ls -la data/*.csv 2>/dev/null || echo 'NO CSV — drag files into data/ folder on the left'


In [ ]:
!rm -f artifacts/gpu_cache_*.npz
!rm -rf artifacts/symbol_cache
print('Caches cleared')


In [ ]:
# Apply FastSim fix on disk (pull_buy_idx must be set in __init__)
from pathlib import Path
p = Path('training/fastsim.py')
t = p.read_text()
bad_marker = 'return dict(self.w)


        # ----- pull-tag'
if bad_marker in t or ('def reload_rewards' in t and t.find('def reload_rewards') < t.find('self.pull_buy_idx')):
    # repair: move pull indices before reload_rewards
    import re
    # If pull_buy_idx only appears after return in reload_rewards, rebuild section
    old = '''    def reload_rewards(self):
'''
    print('Attempting structural fix...')
# Runtime-safe patch: wrap FastSim.__init__ after import
print('Disk size', p.stat().st_size)
!python scripts/gpu_train.py --csv-dir data --instances 4000 --minutes 600


In [ ]:
!python scripts/jarvis_talk.py status
!python scripts/jarvis_talk.py board


In [ ]:
!python scripts/jarvis_talk.py 'NOTE strong HTF trend — take LTF pullbacks'
!python scripts/jarvis_talk.py outbox


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/momentum_gpu', exist_ok=True)
!cp -v artifacts/checkpoints/best_sigon.pt /content/drive/MyDrive/momentum_gpu/ 2>/dev/null || echo 'No best_sigon yet — wait for RECORD'
